# Fuzzy learning-degree smoke test

Generated thin entry point. All method logic lives in `src/`.

In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path

REPO = Path(os.environ["NEURON_DEATH_SOURCE"])
DATA = Path(os.environ["NEURON_DEATH_DATA"])
assert (REPO / "src/train.py").is_file()
assert (DATA / "mnist.npz").is_file()
os.chdir(REPO)
print("CUDA devices:", __import__("torch").cuda.device_count(), flush=True)
assert __import__("torch").cuda.device_count() == 2, "This job requires two independent T4 GPUs"
subprocess.run([sys.executable, "-m", "pytest", "tests", "-q",
                "-p", "no:cacheprovider"], cwd=REPO, check=True)

configs = sorted(glob.glob(str(REPO / "configs/fuzzy_smoke/*.json")))
assert len(configs) == 2, f"expected 2 smoke configs, found {len(configs)}"
RUNS = Path("/kaggle/working/fuzzy_smoke_runs")
subprocess.run([sys.executable, "scripts/launch_pair.py", *configs,
                "--gpus", "0,1", "--runs-root", str(RUNS),
                "--data-root", str(DATA), "--budget-hours", "1"],
               cwd=REPO, check=True)
for config in configs:
    run_id = __import__("json").loads(Path(config).read_text())["run_id"]
    summary = __import__("json").loads((RUNS / run_id / "summary.json").read_text())
    assert summary["status"] == "complete"
    assert (RUNS / run_id / "learning_degree.parquet").is_file()
print("FUZZY_SMOKE_VALIDATED", flush=True)